In [84]:
from pathlib import Path
import json
import math
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
from IPython.display import display

from sklearn.metrics import roc_auc_score
from MLstatkit.stats import Delong_test
import h2o

In [85]:
data_dir = Path("../data/machine_learning")
models_root = Path("../output/models")
out_dir = Path("../output/machine_learning")
out_dir.mkdir(parents=True, exist_ok=True)

calib_dir = out_dir / "calibration"
calib_dir.mkdir(parents=True, exist_ok=True)

In [86]:
target = "outcome_lead"

model_specs = [
    # 1) OneICU every minute MBP model (all features; includes invasive_mbp_00..59)
    {
        "dataset": "oneicu",
        "model_tag": "all_features",
        "label": "OneICU 1-minute MAP model",
        "color": "#6CC5B0",
    },
    # 2) OneICU every 5-minute MBP model
    {
        "dataset": "oneicu",
        "model_tag": "mbp_5min_only",
        "label": "OneICU 5-minute MAP model",
        "color": "#9CD7C7",
    },
    # 3) OneICU hourly MBP model (mbp_00 only among invasive_mbp_XX)
    {
        "dataset": "oneicu",
        "model_tag": "mbp_00_only",
        "label": "OneICU hourly MAP model",
        "color": "#C7E8DF",
    },
    # 4) eICU (every 5-minute MBP model)
    {
        "dataset": "eicu",
        "model_tag": "all_features",
        "label": "eICU (5-minute MAP model)",
        "color": "#FF8AB7",
    },
    # 5) MIMIC-IV (hourly MBP model)
    {
        "dataset": "mimiciv",
        "model_tag": "all_features",
        "label": "MIMIC-IV (hourly MAP model)",
        "color": "#4269D0",
    },
]

In [87]:
FIG_W, FIG_H, FIG_DPI = 3.35, 3.35, 600
plt.rcParams.update({
    "font.size": 7,
    "axes.titlesize": 7,
    "axes.labelsize": 7,
    "legend.fontsize": 6,
    "xtick.labelsize": 6,
    "ytick.labelsize": 6,
    "axes.grid": False,
})

In [88]:
h2o.init()
h2o.no_progress()

Checking whether there is an H2O instance running at http://localhost:54321. connected.
Please download and install the latest version from: https://h2o-release.s3.amazonaws.com/h2o/latest_stable.html


H2O_cluster_uptime:,1 hour 28 mins
H2O_cluster_timezone:,Asia/Tokyo
H2O_data_parsing_timezone:,UTC
H2O_cluster_version:,3.46.0.7
H2O_cluster_version_age:,10 months and 19 days
H2O_cluster_name:,H2O_from_python_kinoshitatakashihiroshi_kx01bz
H2O_cluster_total_nodes:,1
H2O_cluster_free_memory:,3.826 Gb
H2O_cluster_total_cores:,8
H2O_cluster_allowed_cores:,8
H2O_cluster_status:,"locked, healthy"


In [89]:
def load_manifest(dataset: str, model_tag: str) -> dict:
    """
    Prefer: models_root/dataset/model_tag/manifest.json
    Fallback (legacy): models_root/dataset/manifest.json (only when model_tag == 'all_features')
    """
    tagged = models_root / dataset / model_tag / "manifest.json"
    legacy = models_root / dataset / "manifest.json"

    if tagged.exists():
        mpath = tagged
    elif model_tag == "all_features" and legacy.exists():
        mpath = legacy
    else:
        raise FileNotFoundError(
            f"Missing manifest for dataset={dataset}, tag={model_tag}. "
            f"Looked for {tagged} (and legacy {legacy})."
        )

    with open(mpath, "r", encoding="utf-8") as f:
        return json.load(f)

In [90]:
def load_leader_and_features(dataset: str, model_tag: str):
    """Return (leader_model, feature_list)."""
    man = load_manifest(dataset, model_tag)
    leader = h2o.load_model(man["leader_model_path"])
    feats = man.get("features", [])
    if not feats:
        raise ValueError(f"No features found in manifest for dataset={dataset}, tag={model_tag}")
    return leader, feats

In [91]:
def get_test_raw(dataset: str):
    test_path = data_dir / f"ml_{dataset}_test.csv"
    if not test_path.exists():
        raise FileNotFoundError(f"Missing test CSV for {dataset}: {test_path}")
    return h2o.import_file(test_path.as_posix())

In [92]:
def prepare_test_view(test_raw, feats: list[str], target: str):
    # validate all needed columns exist
    missing = [c for c in feats if c not in test_raw.columns]
    if missing:
        raise ValueError(f"Test frame missing training features (first few): {missing[:10]}")

    # slice to feats + target
    test_use = test_raw[:, feats + [target]]

    # enforce types
    for c in feats:
        test_use[c] = test_use[c].asnumeric()
    test_use[target] = test_use[target].asfactor()
    return test_use

In [93]:
def y_true_from_test(test, target: str) -> np.ndarray:
    """y_true as integers 0/1 (handles string factors '0'/'1')."""
    y_true = test[target].as_data_frame(use_pandas=True, use_multi_thread=True)[target]
    y_true = pd.to_numeric(y_true, errors="coerce").astype(int).values
    return y_true

In [94]:
def y_score_from_model(model, test) -> np.ndarray:
    """Predicted probability for positive class (p1)."""
    pred = model.predict(test)
    y_score = pred["p1"].as_data_frame(use_pandas=True, use_multi_thread=True).values.ravel().astype(float)
    # remove prediction frame to keep cluster memory stable
    h2o.remove(pred)
    return y_score

In [95]:
pred_cache_dir = out_dir / "pred_cache_npz"
pred_cache_dir.mkdir(parents=True, exist_ok=True)

In [96]:
def cache_path(dataset: str, model_tag: str) -> Path:
    safe_tag = model_tag.replace("/", "_")
    return pred_cache_dir / f"pred_{dataset}_{safe_tag}.npz"

In [97]:
_test_raw_cache = {}
predictions = []

In [98]:
for spec in model_specs:
    ds = spec["dataset"]
    tag = spec["model_tag"]
    label = spec["label"]

    npz_path = cache_path(ds, tag)

    if npz_path.exists():
        arr = np.load(npz_path, allow_pickle=False)
        y_true = arr["y_true"].astype(int)
        y_score = arr["y_score"].astype(float)
        predictions.append({**spec, "y_true": y_true, "y_score": y_score, "n": int(y_true.size)})
        print(f"[cache] loaded: {label}  n={y_true.size:,}  ({npz_path.name})")
        continue

    print(f"[run] scoring: {label}")
    leader, feats = load_leader_and_features(ds, tag)

    if ds not in _test_raw_cache:
        _test_raw_cache[ds] = get_test_raw(ds)
    test_raw = _test_raw_cache[ds]
    test_use = prepare_test_view(test_raw, feats, target)

    y_true = y_true_from_test(test_use, target)
    y_score = y_score_from_model(leader, test_use)

    np.savez_compressed(npz_path, y_true=y_true.astype(np.int8), y_score=y_score.astype(np.float32))
    print(f"[cache] saved: {npz_path}")

    predictions.append({**spec, "y_true": y_true, "y_score": y_score, "n": int(y_true.size)})

    # cleanup H2O objects
    h2o.remove(test_use)
    try:
        h2o.remove(leader)
    except Exception:
        pass

[cache] loaded: OneICU 1-minute MAP model  n=539,659  (pred_oneicu_all_features.npz)
[cache] loaded: OneICU 5-minute MAP model  n=539,659  (pred_oneicu_mbp_5min_only.npz)
[cache] loaded: OneICU hourly MAP model  n=539,659  (pred_oneicu_mbp_00_only.npz)
[cache] loaded: eICU (5-minute MAP model)  n=229,628  (pred_eicu_all_features.npz)
[cache] loaded: MIMIC-IV (hourly MAP model)  n=799,779  (pred_mimiciv_all_features.npz)


In [99]:
for fr in _test_raw_cache.values():
    try:
        h2o.remove(fr)
    except Exception:
        pass
_test_raw_cache.clear()

# quick sanity check: AUROC values
print("\nQuick AUROC check (sklearn):")
for p in predictions:
    auc = roc_auc_score(p["y_true"], p["y_score"])
    print(f" - {p['label']}: {auc:.6f}  (n={p['n']:,})")


Quick AUROC check (sklearn):
 - OneICU 1-minute MAP model: 0.941518  (n=539,659)
 - OneICU 5-minute MAP model: 0.939102  (n=539,659)
 - OneICU hourly MAP model: 0.900867  (n=539,659)
 - eICU (5-minute MAP model): 0.898934  (n=229,628)
 - MIMIC-IV (hourly MAP model): 0.799306  (n=799,779)


In [100]:
def calibration_table(y_true: np.ndarray, y_prob: np.ndarray, n_bins: int = 10, strategy: str = "quantile") -> pd.DataFrame:
    """
    Build a reliability table with columns:
      bin, n, mean_pred, frac_pos
    strategy:
      - 'quantile': equal-count bins by predicted probability quantiles
      - 'uniform' : equal-width bins in [0,1]
    """
    y_true = np.asarray(y_true).astype(int)
    y_prob = np.asarray(y_prob).astype(float)

    eps = 1e-6
    y_prob = np.clip(y_prob, eps, 1 - eps)

    if strategy == "quantile":
        qs = np.linspace(0, 1, n_bins + 1)
        edges = np.quantile(y_prob, qs)
        # handle duplicates (common when many identical probs)
        edges = np.unique(edges)
        if edges.size < 3:  # too few bins; fallback
            strategy = "uniform"

    if strategy == "uniform":
        edges = np.linspace(0.0, 1.0, n_bins + 1)

    # pandas cut needs strictly increasing bin edges
    edges = np.asarray(edges)
    if np.any(np.diff(edges) <= 0):
        edges = np.linspace(0.0, 1.0, n_bins + 1)

    bins = pd.cut(y_prob, bins=edges, include_lowest=True, duplicates="drop")
    df = pd.DataFrame({"y": y_true, "p": y_prob, "bin": bins})
    g = df.groupby("bin", observed=True)
    out = g.agg(
        n=("y", "size"),
        mean_pred=("p", "mean"),
        frac_pos=("y", "mean"),
    ).reset_index(drop=False)

    # add a simple numeric bin index for plotting
    out["bin_idx"] = np.arange(len(out))
    return out

In [101]:
def brier_score(y_true: np.ndarray, y_prob: np.ndarray) -> float:
    y_true = np.asarray(y_true).astype(float)
    y_prob = np.asarray(y_prob).astype(float)
    return float(np.mean((y_prob - y_true) ** 2))

In [102]:
def plot_calibration_curve(cal_df: pd.DataFrame, title: str, out_path: Path, model_color: str):
    fig, ax = plt.subplots(figsize=(FIG_W, FIG_H))
    ax.plot([0, 1], [0, 1], linestyle="--", linewidth=0.8, color="0.5", label="Perfect calibration")
    ax.plot(
        cal_df["mean_pred"], cal_df["frac_pos"],
        marker="o", linewidth=0.9, markersize=3,
        color=model_color,  # UPDATED
        label="Model"
    )
    ax.set_xlim(0, 1)
    ax.set_ylim(0, 1)
    ax.set_xlabel("Mean predicted probability")
    ax.set_ylabel("Observed event rate")
    ax.set_title(title)
    ax.legend(loc="upper left", frameon=False)
    fig.tight_layout()
    fig.savefig(out_path, dpi=FIG_DPI, format="tiff")
    plt.close(fig)

In [103]:
calib_rows = []

In [104]:
for p in predictions:
    y_true = p["y_true"]
    y_score = p["y_score"]

    cal_df = calibration_table(y_true, y_score, n_bins=10, strategy="quantile")
    bs = brier_score(y_true, y_score)
    prev = float(np.mean(y_true))

    out_path = calib_dir / f"calibration_{p['dataset']}_{p['model_tag']}.tiff"
    plot_calibration_curve(cal_df, p["label"], out_path, model_color=p["color"])

    calib_rows.append({
        "model": p["label"],
        "dataset": p["dataset"],
        "model_tag": p["model_tag"],
        "n": int(y_true.size),
        "prevalence": prev,
        "brier_score": bs,
        "n_bins": int(len(cal_df)),
        "calibration_plot": out_path.as_posix(),
    })

    print(f"Saved: {out_path.name} | prevalence={prev:.4f} | brier={bs:.6f}")

Saved: calibration_oneicu_all_features.tiff | prevalence=0.1282 | brier=0.053162
Saved: calibration_oneicu_mbp_5min_only.tiff | prevalence=0.1282 | brier=0.054163
Saved: calibration_oneicu_mbp_00_only.tiff | prevalence=0.1282 | brier=0.066790
Saved: calibration_eicu_all_features.tiff | prevalence=0.1136 | brier=0.064846
Saved: calibration_mimiciv_all_features.tiff | prevalence=0.1197 | brier=0.087379


In [105]:
calib_metrics_df = pd.DataFrame(calib_rows)
calib_metrics_csv = out_dir / "calibration_metrics.csv"
calib_metrics_df.to_csv(calib_metrics_csv, index=False)

In [106]:
display(calib_metrics_df)
print(f"\nSaved calibration metrics: {calib_metrics_csv}")

,model,dataset,model_tag,n,prevalence,brier_score,n_bins,calibration_plot
0,OneICU 1-minute MAP model,oneicu,all_features,539659,0.128153,0.053162,10,../output/machine_learning/calibration/calibra...
1,OneICU 5-minute MAP model,oneicu,mbp_5min_only,539659,0.128153,0.054163,10,../output/machine_learning/calibration/calibra...
2,OneICU hourly MAP model,oneicu,mbp_00_only,539659,0.128153,0.066790,10,../output/machine_learning/calibration/calibra...
3,eICU (5-minute MAP model),eicu,all_features,229628,0.113632,0.064846,10,../output/machine_learning/calibration/calibra...
4,MIMIC-IV (hourly MAP model),mimiciv,all_features,799779,0.119676,0.087379,10,../output/machine_learning/calibration/calibra...



Saved calibration metrics: ../output/machine_learning/calibration_metrics.csv


In [107]:
def se_from_ci(ci: tuple[float, float]) -> float:
    """Approximate SE from a symmetric normal CI width (CI assumed ~ auc ± z*SE)."""
    lo, hi = float(ci[0]), float(ci[1])
    return max(0.0, (hi - lo) / (2.0 * 1.96))

In [108]:
def mlstatkit_delong(true: np.ndarray,
                     prob_A: np.ndarray,
                     prob_B: np.ndarray,
                     verbose: int = 0):
    """Run MLstatkit Delong_test with compatibility across versions.

    Returns a dict with:
      z, p, auc_A, auc_B, ci_A, ci_B, info(dict)
    """
    true = np.asarray(true).astype(int)
    prob_A = np.asarray(prob_A).astype(float)
    prob_B = np.asarray(prob_B).astype(float)

    # Newer API (some versions): Delong_test(true, prob_A, prob_B, alpha=0.95, return_ci=True, return_auc=True, verbose=0)
    try:
        ret = Delong_test(true, prob_A, prob_B,
                          alpha=0.95,
                          return_ci=True,
                          return_auc=True,
                          verbose=verbose)
    except TypeError:
        # Older API: Delong_test(true, prob_A, prob_B, return_ci=True, alpha=0.95)
        ret = Delong_test(true, prob_A, prob_B, return_ci=True, alpha=0.95)

    # Unpack robustly
    z = p = None
    ci_A = ci_B = None
    auc_A = auc_B = None
    info = {}

    if isinstance(ret, tuple):
        if len(ret) == 2:
            z, p = ret
        elif len(ret) == 4:
            z, p, ci_A, ci_B = ret
        elif len(ret) == 6:
            z, p, ci_A, ci_B, auc_A, auc_B = ret
        elif len(ret) >= 7:
            z, p, ci_A, ci_B, auc_A, auc_B, info = ret[:7]
        else:
            raise RuntimeError(f"Unexpected Delong_test return length: {len(ret)}")
    else:
        raise RuntimeError(f"Unexpected Delong_test return type: {type(ret)}")

    # If AUCs are not returned by this MLstatkit version, compute them
    if auc_A is None:
        auc_A = roc_auc_score(true, prob_A)
    if auc_B is None:
        auc_B = roc_auc_score(true, prob_B)

    # If CIs are not returned, raise (we need them for reporting)
    if ci_A is None or ci_B is None:
        raise RuntimeError(
            "Delong_test did not return CIs. Please upgrade MLstatkit to >=0.1.8, "
            "or ensure return_ci=True is supported."
        )

    return {
        "z": float(z),
        "p": float(p),
        "auc_A": float(auc_A),
        "auc_B": float(auc_B),
        "ci_A": (float(ci_A[0]), float(ci_A[1])),
        "ci_B": (float(ci_B[0]), float(ci_B[1])),
        "info": dict(info) if isinstance(info, dict) else {},
    }

In [109]:
pred_by_label = {p["label"]: p for p in predictions}

In [110]:
def prevalence_baseline_probs(y_true: np.ndarray) -> np.ndarray:
    """Constant-probability baseline equal to outcome prevalence."""
    y_true = np.asarray(y_true).astype(int)
    prev = float(np.mean(y_true))
    return np.full(y_true.shape[0], prev, dtype=float)

In [111]:
auroc_rows = []
model_stats = {}

In [112]:
for p in predictions:
    y = p["y_true"]
    s = p["y_score"]
    base = prevalence_baseline_probs(y)

    res = mlstatkit_delong(y, s, base, verbose=0)

    auc = res["auc_A"]
    ci = res["ci_A"]
    se = se_from_ci(ci)

    n_pos = int(np.sum(y == 1))
    n_neg = int(np.sum(y == 0))
    method = res.get("info", {}).get("method", "delong_or_bootstrap")  # may be absent in older versions

    auroc_rows.append({
        "model": p["label"],
        "dataset": p["dataset"],
        "model_tag": p["model_tag"],
        "n_total": int(p.get("n_total", y.size)),
        "n_used": int(p.get("n_used", y.size)),
        "n_pos": n_pos,
        "n_neg": n_neg,
        "auroc": float(auc),
        "ci95_lo": float(ci[0]),
        "ci95_hi": float(ci[1]),
        "se_from_ci": float(se),
        "mlstatkit_method": method,
        "ci_source": "MLstatkit Delong_test (vs prevalence baseline)",
    })

    model_stats[p["label"]] = {
        "auc": float(auc),
        "ci": (float(ci[0]), float(ci[1])),
        "se": float(se),
        "dataset": p["dataset"],
    }

In [113]:
auroc_df = pd.DataFrame(auroc_rows).sort_values("auroc", ascending=False)
auroc_csv = out_dir / "auroc_delong_ci.csv"
auroc_df.to_csv(auroc_csv, index=False)

In [114]:
display(auroc_df)
print(f"Saved: {auroc_csv}")

,model,dataset,model_tag,n_total,n_used,n_pos,n_neg,auroc,ci95_lo,ci95_hi,se_from_ci,mlstatkit_method,ci_source
0,OneICU 1-minute MAP model,oneicu,all_features,539659,539659,69159,470500,0.941518,0.940598,0.942438,0.000469,delong,MLstatkit Delong_test (vs prevalence baseline)
1,OneICU 5-minute MAP model,oneicu,mbp_5min_only,539659,539659,69159,470500,0.939102,0.938158,0.940045,0.000481,delong,MLstatkit Delong_test (vs prevalence baseline)
2,OneICU hourly MAP model,oneicu,mbp_00_only,539659,539659,69159,470500,0.900867,0.899570,0.902165,0.000662,delong,MLstatkit Delong_test (vs prevalence baseline)
3,eICU (5-minute MAP model),eicu,all_features,229628,229628,26093,203535,0.898934,0.896926,0.900942,0.001025,delong,MLstatkit Delong_test (vs prevalence baseline)
4,MIMIC-IV (hourly MAP model),mimiciv,all_features,799779,799779,95714,704065,0.799306,0.797705,0.800906,0.000816,delong,MLstatkit Delong_test (vs prevalence baseline)


Saved: ../output/machine_learning/auroc_delong_ci.csv


In [115]:
oneicu_1m = "OneICU 1-minute MAP model"

p_rows = []

# Correlated comparisons (same OneICU test set)
y_oneicu = pred_by_label[oneicu_1m]["y_true"]
s_oneicu_1m = pred_by_label[oneicu_1m]["y_score"]

In [117]:
for other in ["OneICU 5-minute MAP model", "OneICU hourly MAP model"]:
    s_other = pred_by_label[other]["y_score"]

    res = mlstatkit_delong(y_oneicu, s_oneicu_1m, s_other, verbose=0)

    p_rows.append({
        "comparison": f"{oneicu_1m} vs {other}",
        "assumption": "correlated (same OneICU test set; MLstatkit DeLong)",
        "auc_1": res["auc_A"],
        "ci95_1": res["ci_A"],
        "auc_2": res["auc_B"],
        "ci95_2": res["ci_B"],
        "diff": res["auc_A"] - res["auc_B"],
        "z_score": res["z"],
        "p_value": res["p"],
        "mlstatkit_method": res.get("info", {}).get("method", "delong_or_bootstrap"),
    })

In [118]:
p_df = pd.DataFrame(p_rows)
p_csv = out_dir / "auroc_comparisons_pvalues.csv"
p_df.to_csv(p_csv, index=False)

display(p_df)
print(f"Saved: {p_csv}")

,comparison,assumption,auc_1,ci95_1,auc_2,ci95_2,diff,z_score,p_value,mlstatkit_method
0,OneICU 1-minute MAP model vs OneICU 5-minute M...,correlated (same OneICU test set; MLstatkit De...,0.941518,"(0.9405979981272178, 0.9424378983167435)",0.939102,"(0.9381583259936433, 0.940044765716093)",0.002416,-23.522785,2.384822e-122,delong
1,OneICU 1-minute MAP model vs OneICU hourly MAP...,correlated (same OneICU test set; MLstatkit De...,0.941518,"(0.9405979981272178, 0.9424378983167435)",0.900867,"(0.8995697596182416, 0.9021651721263918)",0.040650,-85.903967,0.000000e+00,delong


Saved: ../output/machine_learning/auroc_comparisons_pvalues.csv
